In [1]:
from pyspark.sql import SparkSession, functions as F
from IPython.display import display

spark = (
    SparkSession.builder
    .appName("OSIPTEL-Modelo-Base-42")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

BASE = "hdfs:///datalake/osiptel_proyecto"

silver = spark.read.parquet(
    f"{BASE}/silver/conexiones_osiptel_depuradas"
)

periodos_revision = spark.read.parquet(
    f"{BASE}/gold/periodos_en_revision"
)

print("Registros Silver:", silver.count())

print("\nESQUEMA:")
silver.printSchema()

print("\nPERIODOS EN REVISIÓN:")
periodos_revision.select(
    "anio", "trimestre"
).orderBy(
    "anio", "trimestre"
).show()

26/10/11 13:12:16 WARN Utils: Your hostname, debian resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/11 13:12:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 13:12:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/11 13:12:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Registros Silver: 128095

ESQUEMA:
root
 |-- mes: date (nullable = true)
 |-- empresa: string (nullable = true)
 |-- tecnologia: string (nullable = true)
 |-- segmento: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- rango_velocidad: string (nullable = true)
 |-- conexiones: long (nullable = true)
 |-- anio: integer (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- fuente_excel: string (nullable = true)
 |-- version_excel: string (nullable = true)


PERIODOS EN REVISIÓN:
+----+---------+
|anio|trimestre|
+----+---------+
|2023|        1|
|2025|        1|
+----+---------+



In [3]:
from pyspark.sql import functions as F
cierres = silver.filter(
    (F.col("anio") >= 2019) &
    F.month("mes").isin(3, 6, 9, 12)
)
cierres_validos = cierres.join(
    periodos_revision.select(
        "anio", "trimestre"
    ).distinct(),
    ["anio", "trimestre"],
    "left_anti"
)
resumen = (
    cierres_validos
    .groupBy("anio", "trimestre")
    .agg(
        F.countDistinct("departamento").alias("departamentos"),
        F.countDistinct("mes").alias("fechas_cierre"),
        F.sum("conexiones").alias("conexiones_totales")
    )
    .orderBy("anio", "trimestre")
)
print("TRIMESTRES DISPONIBLES:")
resumen.show(40, truncate=False)
print("PERIODOS POR DEPARTAMENTO:")
(
    cierres_validos
    .groupBy("departamento")
    .agg(
        F.countDistinct(
            "anio", "trimestre"
        ).alias("periodos")
    )
    .orderBy("departamento")
    .show(40, truncate=False)
)

TRIMESTRES DISPONIBLES:
+----+---------+-------------+-------------+------------------+
|anio|trimestre|departamentos|fechas_cierre|conexiones_totales|
+----+---------+-------------+-------------+------------------+
|2019|1        |25           |1            |2235850           |
|2019|2        |25           |1            |2439997           |
|2019|3        |25           |1            |2420786           |
|2019|4        |25           |1            |2405657           |
|2020|1        |25           |1            |2438547           |
|2020|2        |25           |1            |2630066           |
|2020|3        |25           |1            |2638737           |
|2020|4        |25           |1            |2776447           |
|2021|1        |25           |1            |2884025           |
|2021|2        |25           |1            |3037349           |
|2021|3        |25           |1            |3078494           |
|2021|4        |25           |1            |3085622           |
|2022|1        |

In [4]:
from pyspark.sql import functions as F
print("TECNOLOGÍAS REGISTRADAS:")
(
    cierres_validos
    .groupBy("tecnologia")
    .agg(F.sum("conexiones").alias("conexiones"))
    .orderBy(F.desc("conexiones"))
    .show(30, truncate=False)
)
panel_trimestral = (
    cierres_validos
    .groupBy("departamento", "anio", "trimestre")
    .agg(
        F.sum("conexiones").alias("conexiones_fijas"),
        F.sum(
            F.when(
                F.upper(F.col("tecnologia")).contains("FIBRA"),
                F.col("conexiones")
            ).otherwise(0)
        ).alias("conexiones_fibra")
    )
    .withColumn(
        "participacion_fibra_pct",
        F.round(
            F.col("conexiones_fibra") * 100.0
            / F.col("conexiones_fijas"),
            4
        )
    )
    .withColumn(
        "indice_trimestre",
        F.col("anio") * 4 + F.col("trimestre")
    )
)
print("\nOBSERVACIONES:", panel_trimestral.count())
print(
    "REGISTROS INVÁLIDOS:",
    panel_trimestral.filter(
        (F.col("conexiones_fijas") <= 0) |
        F.col("participacion_fibra_pct").isNull() |
        (F.col("participacion_fibra_pct") < 0) |
        (F.col("participacion_fibra_pct") > 100)
    ).count()
)
print("\nPARTICIPACIÓN DE FIBRA POR DEPARTAMENTO - 2026 T1:")
(
    panel_trimestral
    .filter(
        (F.col("anio") == 2026) &
        (F.col("trimestre") == 1)
    )
    .select(
        "departamento",
        "conexiones_fijas",
        "conexiones_fibra",
        "participacion_fibra_pct"
    )
    .orderBy("participacion_fibra_pct")
    .show(25, truncate=False)
)

TECNOLOGÍAS REGISTRADAS:
+--------------------+----------+
|tecnologia          |conexiones|
+--------------------+----------+
|3) Cablemodem       |40593597  |
|5) Fibra Óptica     |29847443  |
|7) Otras Tecnologías|13100581  |
|2) xDSL             |3702642   |
|6) Satelital        |443148    |
|4) Wimax            |68616     |
+--------------------+----------+


OBSERVACIONES: 675
REGISTROS INVÁLIDOS: 0

PARTICIPACIÓN DE FIBRA POR DEPARTAMENTO - 2026 T1:
+-------------+----------------+----------------+-----------------------+
|departamento |conexiones_fijas|conexiones_fibra|participacion_fibra_pct|
+-------------+----------------+----------------+-----------------------+
|MADRE DE DIOS|15593           |5799            |37.1898                |
|CUSCO        |106695          |77996           |73.1018                |
|AMAZONAS     |13008           |9554            |73.4471                |
|AREQUIPA     |224313          |166708          |74.3194                |
|LORETO       |41203 

In [5]:
from pyspark.sql import Window, functions as F
ventana = Window.partitionBy("departamento").orderBy("indice_trimestre")
dataset_modelo = (
    panel_trimestral
    .withColumn(
        "indice_objetivo",
        F.lead("indice_trimestre").over(ventana)
    )
    .withColumn(
        "anio_objetivo",
        F.lead("anio").over(ventana)
    )
    .withColumn(
        "trimestre_objetivo",
        F.lead("trimestre").over(ventana)
    )
    .withColumn(
        "fibra_siguiente_pct",
        F.lead("participacion_fibra_pct").over(ventana)
    )
    .withColumn(
        "distancia_trimestres",
        F.col("indice_objetivo") - F.col("indice_trimestre")
    )
    # Solo pares de trimestres realmente consecutivos
    .filter(F.col("distancia_trimestres") == 1)
    .drop("indice_objetivo", "distancia_trimestres")
)
entrenamiento = dataset_modelo.filter(
    F.col("anio_objetivo") <= 2024
)

validacion = dataset_modelo.filter(
    (F.col("anio_objetivo") == 2025) &
    (F.col("trimestre_objetivo").isin(3, 4))
)
prueba = dataset_modelo.filter(
    (F.col("anio_objetivo") == 2026) &
    (F.col("trimestre_objetivo") == 1)
)
print("OBSERVACIONES APTAS:", dataset_modelo.count())
print("ENTRENAMIENTO:", entrenamiento.count())
print("VALIDACIÓN:", validacion.count())
print("PRUEBA:", prueba.count())
print("\nEJEMPLO DE PREDICCIÓN:")
(
    dataset_modelo
    .filter(F.col("departamento") == "LA LIBERTAD")
    .select(
        "departamento",
        "anio",
        "trimestre",
        "participacion_fibra_pct",
        "anio_objetivo",
        "trimestre_objetivo",
        "fibra_siguiente_pct"
    )
    .orderBy(F.desc("indice_trimestre"))
    .show(8, truncate=False)
)

OBSERVACIONES APTAS: 600
ENTRENAMIENTO: 525
VALIDACIÓN: 50
PRUEBA: 25

EJEMPLO DE PREDICCIÓN:
+------------+----+---------+-----------------------+-------------+------------------+-------------------+
|departamento|anio|trimestre|participacion_fibra_pct|anio_objetivo|trimestre_objetivo|fibra_siguiente_pct|
+------------+----+---------+-----------------------+-------------+------------------+-------------------+
|LA LIBERTAD |2025|4        |76.7317                |2026         |1                 |77.8043            |
|LA LIBERTAD |2025|3        |74.9014                |2025         |4                 |76.7317            |
|LA LIBERTAD |2025|2        |73.1782                |2025         |3                 |74.9014            |
|LA LIBERTAD |2024|3        |64.411                 |2024         |4                 |67.834             |
|LA LIBERTAD |2024|2        |58.536                 |2024         |3                 |64.411             |
|LA LIBERTAD |2024|1        |52.0289              

In [6]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F
assembler = VectorAssembler(
    inputCols=["participacion_fibra_pct"],
    outputCol="features"
)
train_ml = assembler.transform(entrenamiento)
val_ml = assembler.transform(validacion)
regresion = LinearRegression(
    featuresCol="features",
    labelCol="fibra_siguiente_pct",
    predictionCol="prediction",
    maxIter=100,
    regParam=0.0
)
modelo_base = regresion.fit(train_ml)
predicciones_val = modelo_base.transform(val_ml)
evaluador_mae = RegressionEvaluator(
    labelCol="fibra_siguiente_pct",
    predictionCol="prediction",
    metricName="mae"
)
evaluador_rmse = RegressionEvaluator(
    labelCol="fibra_siguiente_pct",
    predictionCol="prediction",
    metricName="rmse"
)
mae_modelo = evaluador_mae.evaluate(predicciones_val)
rmse_modelo = evaluador_rmse.evaluate(predicciones_val)
baseline_val = validacion.withColumn(
    "prediction",
    F.col("participacion_fibra_pct")
)
mae_baseline = evaluador_mae.evaluate(baseline_val)
rmse_baseline = evaluador_rmse.evaluate(baseline_val)

print("ENTRENAMIENTO COMPLETADO")
print("\nMODELO: REGRESIÓN LINEAL")
print(f"Coeficiente: {modelo_base.coefficients[0]:.6f}")
print(f"Intercepto: {modelo_base.intercept:.6f}")
print("\nRESULTADOS DE VALIDACIÓN - 2025 T3 Y T4")
print(f"{'Modelo':<25} {'MAE':>10} {'RMSE':>10}")
print("-" * 47)
print(f"{'Persistencia':<25} {mae_baseline:>10.4f} {rmse_baseline:>10.4f}")
print(f"{'Regresión lineal':<25} {mae_modelo:>10.4f} {rmse_modelo:>10.4f}")

ENTRENAMIENTO COMPLETADO

MODELO: REGRESIÓN LINEAL
Coeficiente: 1.037807
Intercepto: 0.248641

RESULTADOS DE VALIDACIÓN - 2025 T3 Y T4
Modelo                           MAE       RMSE
-----------------------------------------------
Persistencia                  1.3703     1.7014
Regresión lineal              2.4769     2.7852


In [7]:
from pyspark.sql import functions as F
diagnostico = (
    predicciones_val
    .select(
        "departamento",
        "anio_objetivo",
        "trimestre_objetivo",
        F.col("fibra_siguiente_pct").alias("valor_real"),
        F.col("participacion_fibra_pct").alias("persistencia"),
        F.col("prediction").alias("regresion")
    )
    .withColumn(
        "error_regresion",
        F.col("regresion") - F.col("valor_real")
    )
    .withColumn(
        "error_persistencia",
        F.col("persistencia") - F.col("valor_real")
    )
)
print("ERROR POR TRIMESTRE DE VALIDACIÓN:")
(
    diagnostico
    .groupBy("anio_objetivo", "trimestre_objetivo")
    .agg(
        F.round(
            F.avg(F.abs("error_regresion")), 4
        ).alias("MAE_regresion"),

        F.round(
            F.avg(F.abs("error_persistencia")), 4
        ).alias("MAE_persistencia")
    )
    .orderBy("anio_objetivo", "trimestre_objetivo")
    .show()
)
print("\n5 MAYORES ERRORES DE LA REGRESIÓN:")
(
    diagnostico
    .withColumn(
        "error_absoluto",
        F.abs("error_regresion")
    )
    .select(
        "departamento",
        "anio_objetivo",
        "trimestre_objetivo",
        F.round("valor_real", 2).alias("real"),
        F.round("regresion", 2).alias("predicho"),
        F.round("error_regresion", 2).alias("error")
    )
    .orderBy(F.desc("error_absoluto"))
    .show(5, truncate=False)
)

ERROR POR TRIMESTRE DE VALIDACIÓN:
+-------------+------------------+-------------+----------------+
|anio_objetivo|trimestre_objetivo|MAE_regresion|MAE_persistencia|
+-------------+------------------+-------------+----------------+
|         2025|                 3|        2.322|          1.5132|
|         2025|                 4|       2.6318|          1.2274|
+-------------+------------------+-------------+----------------+


5 MAYORES ERRORES DE LA REGRESIÓN:
+------------+-------------+------------------+-----+--------+-----+
|departamento|anio_objetivo|trimestre_objetivo|real |predicho|error|
+------------+-------------+------------------+-----+--------+-----+
|LORETO      |2025         |4                 |79.76|84.94   |5.18 |
|AMAZONAS    |2025         |4                 |75.67|80.1    |4.44 |
|SAN MARTIN  |2025         |3                 |91.3 |95.6    |4.31 |
|UCAYALI     |2025         |4                 |90.03|94.15   |4.12 |
|AMAZONAS    |2025         |3                 |76

## Conclusiones del modelo base — Apartado 4.2

Se entrenó un modelo de regresión lineal mediante PySpark MLlib
para predecir la participación de fibra óptica por departamento
en el siguiente trimestre.

Se utilizaron 525 observaciones para entrenamiento y 50 para
validación, con una partición cronológica para evitar utilizar
información futura durante el entrenamiento.

### Resultados de validación
| Modelo | MAE | RMSE |
|---|---:|---:|
| Persistencia | 1.3703 | 1.7014 |
| Regresión lineal | 2.4769 | 2.7852 |

Las métricas se expresan en puntos porcentuales de participación
de fibra óptica.

La regresión lineal presentó errores superiores al método de
persistencia en ambos trimestres evaluados: 2025-T3 y 2025-T4.

Los mayores errores identificados corresponden a sobreestimaciones,
especialmente en Loreto, Amazonas, San Martín y Ucayali.

Por tanto, se establece un MAE de 2.4769 como línea base técnica
del primer modelo supervisado. La persistencia, con un MAE de
1.3703, representa la referencia sencilla que las mejoras
posteriores deberán intentar superar.

### Mejoras propuestas

En el apartado 4.3 se evaluarán variables históricas adicionales,
un algoritmo alternativo y ajustes de hiperparámetros.

La selección del modelo definitivo se realizará mediante métricas
de validación, manteniendo reservado el conjunto de prueba
2026-T1 para su evaluación final.